# Crypto Portfolio Analytics

Discovers live perp positions and spot balances in configured Hyperliquid wallets, uses Hyperliquid daily candles for perp markets and CoinGecko prices and market data for supported assets, then calculates correlations and constrained long/short full-Kelly weights and reports portfolio performance, risk, and factor estimates. Implementation lives in `crypto_portfolio_analytics_implementation.py`.


## Load implementation

Set the wallet addresses to analyze, then load the companion implementation and create the analytics object.


In [1]:
from crypto_portfolio_analytics_implementation import CryptoPortfolioAnalytics

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

analysis=CryptoPortfolioAnalytics(data_dir='data', wallet_addresses=WALLET_ADDRESSES)


## Market data refresh

Discovers current nonzero perp positions and spot balances. Perp prices come from Hyperliquid daily candles (including HIP-3 markets); CoinGecko supplies spot-only prices and market-cap/volume data without requiring an API key. Assets without enough common daily returns against BTC are reported and skipped. Edit `WALLET_ADDRESSES` above to select wallets.


In [2]:
analysis.refresh_market_data()


CoinGecko rate limit (HTTP 429); retrying attempt 2/8 in 60.0 seconds.
Refreshed saved prices and market data: IMX 366 days, FOGO 327 days, USDC 2 days, BTC 366 days, ZRO 366 days, RSR 366 days, PENGU 366 days, MELANIA 366 days, HYPE 2 days, AVAX 366 days, GRASS 366 days, TRUMP 366 days, INIT 366 days, LIT 316 days, USELESS 56 days, ETH 366 days, TRX 366 days, ADA 366 days, AERO 366 days, xyz:GOLD 316 days, xyz:AAPL 347 days, xyz:GOOGL 350 days, xyz:NET 77 days, xyz:RDDT 77 days, xyz:JIO 11 days, io:PRL 20 days. Assets discovered from 2 wallet(s): AVAX, IMX, GRASS, TRUMP, INIT, FOGO, LIT, USELESS, USDC, BTC, ETH, TRX, ZRO, ADA, RSR, PENGU, MELANIA, AERO, xyz:GOLD, xyz:AAPL, xyz:GOOGL, xyz:NET, xyz:RDDT, HYPE. Public historical requests are limited to the latest 365 days; older saved rows were retained.
Skipping spot-only wallet asset(s) without a unique CoinGecko ID: MAX (ambiguous CoinGecko symbol).
Using Hyperliquid candles for perp asset(s) without a CoinGecko mapping: AVAX, GRASS, 

## Price history

Loads assets discovered from wallet positions and spot balances, plus BTC for benchmark comparisons.


In [3]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [4]:
analysis.calculate_returns()


## Correlation heatmap

Displays the daily return correlation matrix as an interactive Plotly heatmap.


In [5]:
analysis.plot_correlations()


## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [6]:
analysis.optimize_portfolio()


Constrained long/short full-Kelly weights, sorted by coin weight (%)
Gross long exposure: 150.0%; gross short exposure: 196.2%; net exposure: -46.2%.
Expected price PnL is a historical mean-forward-return contribution per $100 of portfolio equity; it excludes funding, fees, and slippage.


,Weight (%),Expected Price PnL / $100 equity (5d)
AVAX,15.00,$-0.087
GRASS,15.00,$0.345
INIT,15.00,$0.217
xyz:NET,15.00,$0.033
AERO,15.00,$0.041
RSR,15.00,$-0.178
ZRO,15.00,$0.135
BTC,15.00,$-0.046
xyz:AAPL,15.00,$0.058
HYPE,15.00,$0.392


## Portfolio performance and risk metrics

Displays the Plotly portfolio-vs-BTC chart and performance and risk tables. Return and volatility annualization use 365 calendar days. Arithmetic annualized return is the daily mean multiplied by 365, not a forecast or compounded return. Historical VaR is an empirical quantile; Normal VaR is a Gaussian-model comparison. The observation and tail counts show how limited the 99% historical estimate is.


In [7]:
analysis.analyze_performance()


,Return
1D,-1.77%
7D,15.16%
M (1 month),50.53%
Q (3 months),N/A
Since Inception,150.60%


,Daily return
Average daily return,1.74%
Largest daily gain,10.56%
Largest daily loss,-3.33%


,All-Time High ($),Minimum Portfolio Value ($),Max Drawdown ($),Max Drawdown (%)
Since inception,$262.73,$100.00,-$14.35,-6.75%


,"Sharpe (excess, sqrt 365)","Sharpe (excess, sqrt 252)","Sharpe (raw, sqrt 252)",Sortino (365d),Beta,Alpha,"Arithmetic Return (365d, linear)",Hist Vol (daily),"Hist Vol (annualized, 365d)",Historical 1D VaR 95%,Historical VaR 95% ($ / $100),Historical 1D VaR 99%,Historical VaR 99% ($ / $100),Normal 1D VaR 95%,Normal VaR 95% ($ / $100),Normal 1D VaR 99%,Normal VaR 99% ($ / $100),Normal 1D VaR 99% (zero mean),"Normal VaR 99% ($ / $100, zero mean)",Daily Return Observations,Historical VaR 99% Tail Observations,Risk-Free Rate,Max Drawdown
In-sample: weights fit and evaluated on same window,9.393,7.804,7.865,25.858,-0.797,632.04%,636.36%,3.52%,67.23%,3.05%,$3.05,3.23%,$3.23,4.04%,$4.04,6.44%,$6.44,8.19%,$8.19,55,1,5.00%,-6.75%


Risk estimates use 55 daily observations and 365-day crypto annualization. Historical 99% VaR has only 1 observed tail observation(s); treat it as a weak tail estimate. Normal VaR is a Gaussian-model comparison, not a tail guarantee. Arithmetic return is the daily mean multiplied by 365, not a forecast or compounded return. These are in-sample results: Kelly weights were optimized and evaluated on the same return window; use a walk-forward backtest for out-of-sample evidence.


## CoinGecko factor model

Fits factor exposures and displays the annual factor estimates and model fit.


In [8]:
analysis.fit_factor_model()


Factor model: R²=16.6% | adjusted R²=15.1% | annual alpha=632.04% | n=55 days
Single-factor BTC beta=-0.797 | factor-model BTC beta=-0.797
CoinGecko cross-sectional factors exclude assets without market-cap history: AVAX, GRASS, TRUMP, INIT, LIT, USELESS, ETH, TRX, ADA, xyz:GOLD, xyz:AAPL, xyz:GOOGL, xyz:NET, xyz:RDDT. Cross-sectional factor model had only 22 complete observations; using the BTC-only model to retain the available portfolio history.


,Beta,HAC t-stat,Factor premium (%/yr),Portfolio contribution (%/yr)
BTC market excess,-0.797,-5.79,0.70%,-0.56%
Alpha,N/A,4.00,N/A,632.04%


,Factor model fit
Observations,55.000000
R-squared,0.166335
Adjusted R-squared,0.150605
Annual alpha,6.320378
BTC beta (factor model),-0.797196
BTC beta (single factor),-0.797196


## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [9]:
analysis.save_outputs()


Saved portfolio, factor, market, and return outputs to data/.
